<a href="https://colab.research.google.com/github/pelinbalci/SLM-FineTune/blob/main/Part_8_1_Unsloth_FineTune_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Part 8: Unsloth FineTune & GGUF

**You've fine-tuned a model. Now what?**

This notebook covers how to actually **use** your fine-tuned model.

---

## What We'll Cover

| Topic | Description |
|-------|-------------|
| Quick Training | Train a model with Unsloth (fast!) |
| Local Inference | Run the model directly |
| Saving Formats | LoRA, merged, GGUF |
| HuggingFace Hub | Upload and share |
| GGUF + llama.cpp | Run on CPU/edge devices |
| Ollama Integration | Easy local deployment |

---

## Deployment Options Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                    YOUR FINE-TUNED MODEL                        │
└─────────────────────────┬───────────────────────────────────────┘
                          │
          ┌───────────────┼───────────────┐
          │               │               │
          ▼               ▼               ▼
   ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
   │   LoRA      │ │   Merged    │ │    GGUF     │
   │  Adapter    │ │   Model     │ │  (llama.cpp)│
   └──────┬──────┘ └──────┬──────┘ └──────┬──────┘
          │               │               │
          ▼               ▼               ▼
   ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
   │ HuggingFace │ │   vLLM /    │ │   Ollama    │
   │ Inference   │ │   TGI       │ │   Local     │
   └─────────────┘ └─────────────┘ └─────────────┘
```

---

# Part A: Setup & Quick Training

Let's train a model quickly with Unsloth so we have something to deploy. See more about thetraining details on: https://medium.com/@balci.pelin/unsloth-vs-standard-training-92d4c35b8ad8

In [ ]:
!pip install -q -U unsloth

In [2]:
import unsloth
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# Configuration - same for both experiments
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LENGTH = 512
BATCH_SIZE = 2
GRAD_ACCUM = 4
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4

# LoRA settings
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Model
model, tokenizer_unsloth = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing
    random_state=42,
)

==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.2.1 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [4]:
# Data
from datasets import load_dataset
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")

from transformers import DataCollatorForSeq2Seq
from transformers import TrainingArguments

def tok_fn(examples):
    out = tokenizer_unsloth(
        examples["text"],
        truncation=True,   # ✅ Cut sequences > MAX_SEQ_LENGTH
        max_length=MAX_SEQ_LENGTH,
        padding=False,          # important: no padding in dataset
    )
    return out

tokenized_dataset = dataset.map(tok_fn, batched=True, remove_columns=dataset.column_names)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
# Data collator with proper truncation
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer_unsloth,
    padding=True,
    pad_to_multiple_of=8,
)

training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    fp16=True,
    bf16=False,
    logging_steps=50,
    save_strategy="no",
    optim="adamw_8bit",
    warmup_ratio=0.03,
    report_to="none",
)

from trl import SFTTrainer

trainer_unsloth = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer_unsloth,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    data_collator=data_collator,
)

result_unsloth = trainer_unsloth.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 2,162,688 of 496,195,456 (0.44% trained)


Step,Training Loss
50,1.886300
100,1.821100


In [6]:
result_unsloth

TrainOutput(global_step=125, training_loss=1.8402185668945312, metrics={'train_runtime': 177.999, 'train_samples_per_second': 5.618, 'train_steps_per_second': 0.702, 'total_flos': 874550236876800.0, 'train_loss': 1.8402185668945312, 'epoch': 1.0})

---

# Part B: Local Inference

The simplest way to use your model: run it directly!

In [7]:
# Enable fast inference mode
FastLanguageModel.for_inference(model)
print("✅ Inference mode enabled")

✅ Inference mode enabled


In [10]:
def chat(prompt, max_tokens=256):
    """Simple chat function."""
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer_unsloth.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer_unsloth.pad_token_id,
    )

    response = tokenizer_unsloth.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

print("✅ chat() function ready")

✅ chat() function ready


In [11]:
# Test the model
print("🧪 Testing the fine-tuned model:\n")

test_prompts = [
    "What is machine learning?",
    "Write a Python function to reverse a string.",
    "Give me 3 tips for better sleep.",
]

for prompt in test_prompts:
    print(f"👤 User: {prompt}")
    response = chat(prompt)
    print(f"🤖 Assistant: {response}")
    print("-" * 50)

🧪 Testing the fine-tuned model:

👤 User: What is machine learning?
🤖 Assistant: Machine learning is a subfield of artificial intelligence that involves the development and application of algorithms for computer systems to learn from data and make predictions or decisions without being explicitly programmed. It aims to enable computers to improve their performance based on experience rather than explicit instructions.

Key components of machine learning include:
- Data: The input features used to train the model.
- Algorithms: Sets of rules or formulas that define how the system should process data.
- Models: Predictions made using trained models.
- Techniques: Methods used in training models, such as regularization and feature selection.

Machine learning techniques can be applied to various tasks, including image recognition, natural language processing, recommendation systems, and autonomous vehicles. Some popular applications include spam filtering, fraud detection, self-driving car

---

# Part C: Saving Your Model

Multiple formats for different use cases:

| Format | Size | Use Case |
|--------|------|----------|
| LoRA adapter | ~10-50 MB | Share adapters, combine with base |
| Merged 16-bit | ~1-14 GB | HuggingFace, full precision |
| Merged 4-bit | ~0.5-4 GB | Smaller, quantized |
| GGUF | ~0.3-8 GB | llama.cpp, Ollama, CPU inference |

## Option 1: Save as GGUF (for llama.cpp / Ollama)

GGUF is the format used by:
- **llama.cpp** — C++ inference engine
- **Ollama** — Easy local LLM runner
- **LM Studio** — Desktop app for LLMs
- **GPT4All** — Local LLM platform

### Quantization Options

| Method | Bits | Size | Quality | Speed |
|--------|------|------|---------|-------|
| q8_0 | 8-bit | Large | Best | Slow |
| q5_k_m | 5-bit | Medium | Great | Medium |
| q4_k_m | 4-bit | Small | Good | Fast |
| q3_k_m | 3-bit | Tiny | OK | Fastest |
| q2_k | 2-bit | Smallest | Poor | Fastest |




Unsloth provides the adapter: save_pretrained_gguf.

In [12]:
import os

# Save as GGUF with different quantizations
GGUF_PATH = "./my_model_gguf"

# q4_k_m is a good balance of size and quality
model.save_pretrained_gguf(
    GGUF_PATH,
    tokenizer_unsloth,
    quantization_method="q4_k_m",
)

print(f"\n✅ GGUF model saved!")

# List GGUF files
print(f"\n📁 GGUF files:")
for f in os.listdir(GGUF_PATH):
    if f.endswith('.gguf'):
        size = os.path.getsize(os.path.join(GGUF_PATH, f)) / 1e9
        print(f"   {f}: {size:.2f} GB")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:09<00:00,  9.57s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:10<00:00, 10.82s/it]


Unsloth: Merge process complete. Saved to `/content/my_model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['./my_model_gguf_gguf/qwen2.5-0.5b-ins

---

# Part D: Upload to HuggingFace Hub

Share your model with the world!

In [13]:
# Login to HuggingFace (you'll need a token)
# Get your token at: https://huggingface.co/settings/tokens

from huggingface_hub import login

# Option 1: Login interactively
login()

# Option 2: Use token directly (uncomment and add your token)
# login(token="hf_your_token_here")

print("💡 To upload, run: login() or login(token='your_token')")

💡 To upload, run: login() or login(token='your_token')


In [15]:
# Upload options (uncomment to use)

HUB_USERNAME = "pelinbalci"  # Change this!
MODEL_NAME = "my-qwen-finetuned"


# Upload GGUF (for Ollama users)
model.push_to_hub_gguf(
    f"{HUB_USERNAME}/{MODEL_NAME}-gguf",
    tokenizer_unsloth,
    quantization_method="q4_k_m",
)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:25<00:00, 25.85s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0.5b-instruct.Q4_K_M.gguf:   4%|4         | 16.8MB /  398MB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/pelinbalci/my-qwen-finetuned-gguf
Unsloth: Cleaning up temporary files...


'pelinbalci/my-qwen-finetuned-gguf'

---

# PART E - Use The Model

1- Ollama

Ollama can pull GGUF models directly from HuggingFace.

2- LM Studio




